# Fluxo agêntico para gerar uma parametrização GoBR

O agente aceita uma pergunta, uma imagem ou um PDF. A saída é uma parametrização na linguagem GoBR.

No Colab, use GPU e adicione `OPENAI_API_KEY` em **Segredos** com acesso habilitado. Em um ambiente local, use a variável de ambiente de mesmo nome. O pedido, o anexo e as referências recuperadas são enviados à API da OpenAI.

In [3]:
%pip install -q "llama-index-core>=0.13,<0.15" "llama-index-embeddings-huggingface>=0.7,<0.8" "sentence-transformers>=3,<6" "transformers>=4.41,<5" "langgraph>=1,<2" "openai>=2,<3"

## Preparar os recursos

Execute uma vez por sessão. Quando necessário, a célula clona o repositório do GitHub e abre `explicacoes.zip` e `manual.zip` automaticamente. Se estiver testando antes de publicar esta versão, envie também `fluxo_gobr.py` pelo painel **Arquivos** do Colab.

In [4]:
import sys
from pathlib import Path
from subprocess import run

repo = Path.cwd()
if not (repo / "explicacoes.zip").is_file():
    repo = repo / "projeto-sistemas-multiagentes"
    if not repo.exists():
        run([
            "git", "clone", "--depth", "1",
            "https://github.com/buenofgustavo/projeto-sistemas-multiagentes.git", str(repo),
        ], check=True)

if not any((pasta / "fluxo_gobr.py").is_file() for pasta in (Path.cwd(), repo)):
    raise FileNotFoundError("Envie fluxo_gobr.py pelo painel Arquivos ou atualize o checkout.")

sys.path[:0] = [str(Path.cwd()), str(repo)]
from fluxo_gobr import Estado, contexto_do_pedido, perguntar, preparar_recursos

MODELO_GPT = "gpt-6-luna"  # Para usar anexos, escolha um modelo com suporte a imagens.
recursos = preparar_recursos(repo, modelo=MODELO_GPT)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (4 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

## Definir o agente

**Leitura do anexo → busca no manual e nos exemplos → geração → revisão.** Se faltar um dado comercial, o fluxo pausa em `esclarecer` e refaz a busca com a resposta. Se a revisão encontrar um problema corrigível, o rascunho volta uma vez para `gerar`.

Edite as instruções nas funções `ler_anexo`, `gerar` e `revisar` para ajustar o comportamento do agente.

In [8]:
from typing import Literal

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel


class LeituraAnexo(BaseModel):
    texto: str


class Proposta(BaseModel):
    acao: Literal["revisar", "esclarecer", "encerrar"]
    texto: str


class Revisao(BaseModel):
    acao: Literal["aprovar", "corrigir", "esclarecer", "encerrar"]
    texto: str


def ler_anexo(estado: Estado):
    if "anexo" not in estado:
        return {"leitura_anexo": ""}
    leitura = recursos.consultar(LeituraAnexo, """
Extraia do anexo as regras comerciais pertinentes ao pedido em português.
Preserve nomes, IDs, valores, unidades, condições, exceções e ordem das regras.
Identifique campos ilegíveis ou ambíguos sem completar as lacunas.
Se o anexo não especificar regras comerciais, indique isso.
""", estado, estado["pergunta"])
    return {"leitura_anexo": leitura.texto}


def gerar(estado: Estado) -> Command[Literal["revisar", "esclarecer", "__end__"]]:
    if not estado["fontes"]:
        return Command(update={"mensagem": "A busca não encontrou referências GoBR."}, goto=END)

    mensagem = (
        f"{contexto_do_pedido(estado)}\n\n"
        f"Rascunho anterior:\n{estado['rascunho']}\n\nParecer:\n{estado['parecer']}"
    )
    proposta = recursos.consultar(Proposta, """
Transforme o pedido comercial e o anexo, se houver, em regras GoBR.
Preserve escopo, IDs, condições, valores, exceções e prioridades. Aplique as alterações do anexo pedidas no texto.
Use os exemplos para confirmar a estrutura e o manual para confirmar a sintaxe.
Use os valores comerciais do pedido, do anexo e dos esclarecimentos.
Se faltar um dado indispensável por ausência, ilegibilidade ou conflito, use 'esclarecer' e faça uma pergunta objetiva.
Se as referências não sustentarem a regra, use 'encerrar' e explique a lacuna em texto.
Se houver um parecer de revisão, corrija os problemas apontados.
Quando puder entregar a regra, use 'revisar' e devolva em texto apenas o GoBR, começando por version 1.0.
""", estado, mensagem)

    if proposta.acao == "revisar":
        return Command(update={"rascunho": proposta.texto}, goto="revisar")
    destino = "esclarecer" if proposta.acao == "esclarecer" else END
    return Command(update={"mensagem": proposta.texto}, goto=destino)


def revisar(estado: Estado) -> Command[Literal["gerar", "esclarecer", "__end__"]]:
    codigo = estado["rascunho"]
    linhas = [linha for linha in codigo.splitlines() if linha.strip() and not linha.lstrip().startswith("#")]
    if len(linhas) < 2 or linhas[0].split() != ["version", "1.0"] or "```" in codigo:
        revisao = Revisao(acao="corrigir", texto="Devolva GoBR puro, com version 1.0 e pelo menos uma regra.")
    else:
        revisao = recursos.consultar(Revisao, """
Revise o rascunho GoBR à luz do pedido, do anexo, dos esclarecimentos e das referências.
Confira escopo, filtros, unidades, exceções, precedência, blocos referenciados e sintaxe.
Se faltar um dado comercial indispensável, use 'esclarecer' e faça uma pergunta objetiva em texto.
Se o problema puder ser corrigido com as informações disponíveis, use 'corrigir' e liste os ajustes em texto.
Se as referências forem insuficientes para confirmar a regra, use 'encerrar' e explique a lacuna em texto.
Use 'aprovar' somente quando a regra estiver completa e coerente, e resuma a revisão em texto.
""", estado, contexto_do_pedido(estado) + f"\n\nRascunho:\n{codigo}")

    if revisao.acao == "aprovar":
        return Command(update={"codigo": codigo}, goto=END)
    if revisao.acao == "corrigir" and not estado["parecer"]:
        return Command(update={"parecer": revisao.texto}, goto="gerar")
    destino = "esclarecer" if revisao.acao == "esclarecer" else END
    return Command(update={"mensagem": revisao.texto}, goto=destino)


def esclarecer(estado: Estado):
    resposta = interrupt(estado["mensagem"])
    turno = {"pergunta": estado["mensagem"], "resposta": resposta}
    return {"historico": [*estado["historico"], turno]}


agente = (
    StateGraph(Estado)
    .add_node(ler_anexo)
    .add_node("buscar_contexto", recursos.buscar_contexto)
    .add_node(gerar)
    .add_node(revisar)
    .add_node(esclarecer)
    .add_edge(START, "ler_anexo")
    .add_edge("ler_anexo", "buscar_contexto")
    .add_edge("buscar_contexto", "gerar")
    .add_edge("esclarecer", "buscar_contexto")
    .compile(checkpointer=InMemorySaver())
)

## Enviar um pedido

- **Só texto**. Use `arquivo=None`.
- **Só anexo**. Use `pergunta=None` e informe o caminho do arquivo.
- **Texto e anexo**. Preencha os dois campos.

Para anexar um PNG, JPG, WebP ou PDF no Colab, envie o arquivo pelo painel **Arquivos** e copie o caminho para `arquivo`.

Se o agente fizer uma pergunta, responda no campo de texto ou pressione Enter para encerrar. Confira a regra no Govendas antes de aplicar.

In [11]:
# pergunta = "Como configurar o atendimento a cada 14 dias para clientes sem venda na empresa, nas carteiras fixas de todos os vendedores?"
# arquivo = repo / "dados/exemplos/RegraInfinittoDesign.pdf"
arquivo = None
pergunta = """Regra comercial da NB Distribuidora

Separar os vendedores em televendas, campo do interior (IDs 27, 35 e 34) e campo da capital (todos os vendedores de campo, exceto esses três IDs). Cada time usa sua carteira fixa.

No televendas, agendar clientes adimplentes a cada 14 dias; ativos para o vendedor e inadimplentes a cada 30 dias; e clientes sem venda a cada 30 dias. Não agendar clientes inativos e inadimplentes.

No campo da capital, agendar clientes sem venda a cada 30 dias; clientes inadimplentes a cada 30 dias; e os demais clientes a cada 14 dias. No campo do interior, agendar todos os clientes da carteira a cada 30 dias.

Aplicar as condições na ordem apresentada.
"""

codigo_gobr = perguntar(agente, pergunta=pergunta, arquivo=arquivo)





['regra_055 (dados/referencias/regra_055.gobr)', 'regra_102 (dados/referencias/regra_102.gobr)', 'manual_0023 (dados/govendas-rules.pdf, páginas PDF 16)', 'manual_0024 (dados/govendas-rules.pdf, páginas PDF 16)', 'manual_0057 (dados/govendas-rules.pdf, páginas PDF 43)']




version 1.0

time "time televendas":
    vendedores todostelevendas

time "time interior":
    vendedores porid("27", "35", "34")

time "time capital":
    vendedores todosdecampo menos ("27", "35", "34")

carteira "carteira televendas":
    clientes carteirafixa("time televendas")

carteira "carteira interior":
    clientes carteirafixa("time interior")

carteira "carteira capital":
    clientes carteirafixa("time capital")

parametros "Regra Padrao da NB Distribuidora Customizavel: 14 dias para adimplentes":
    da carteira("carteira televendas")
    quando adimplentes
    frequencia de 14 dias

parametros "Regra Padrao da NB Distribuidora: 30 dias para ativos inadimplentes":
    da carteira("carteira televend